# TwinCare-Glyco
### A Digital Twin for Near-Term Hyperglycemia Risk in Type 2 Diabetes

This notebook fuses a synthetic-but-clinically-grounded **static EHR profile**
(age, HbA1c, medication, ...) with **dynamic wearable/CGM-style data** (the
`GlucoBench_benchmark_dataset-selected-columns.csv` sample) into a per-patient
**Digital Twin**, forecasts near-term glucose, derives a hyperglycemia-risk
probability from that forecast, and hands the result to a doctor-facing
dashboard (`dashboard/app.py`).

```
Static EHR profile  ---+
                        |--> Data fusion --> Feature engineering --> Forecast model --> Risk score --> Dashboard
Dynamic CGM/wearable ---+
```

**What makes this version different from a typical student notebook:** every
design choice below was made by first measuring what the real data actually
supports, not by assuming a textbook glucose-spike story would work out of
the box. Section 2 shows the audit that changed the whole approach.

All heavy lifting lives in `src/pipeline.py` so the dashboard and this
notebook can never drift apart -- this notebook narrates and visualizes the
same functions the dashboard's artifacts are built from.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pipeline as p

pd.set_option("display.max_columns", 50)
print("Project root:", ROOT)


Project root: C:\Users\karti\OneDrive\Desktop\Healthcare Project


## 1. Load the dynamic (wearable/CGM-style) data

`GlucoBench_benchmark_dataset-selected-columns.csv` contains 10 patients, ~9
days each, at an irregular ~5-15 minute sampling rate: continuous glucose
readings, insulin bolus/basal doses, carbohydrate intake and meal type, and
exercise step counts with an intensity label.


In [2]:
dynamic_df = p.load_dynamic_data()
print(dynamic_df.shape)
dynamic_df.head()


(15731, 10)


,patient_id,timestamp,glucose,cgm_quality_flag,insulin_bolus,insulin_basal,carbs,meal_type,exercise_steps,exercise_intensity
0,U001,2024-09-01 00:00:00,127.0,1,0.0,1.05,0,none,0,none
1,U001,2024-09-01 00:15:00,126.5,1,0.0,0.84,0,none,0,none
2,U001,2024-09-01 00:20:00,126.8,1,0.0,1.19,0,none,0,none
3,U001,2024-09-01 00:25:00,127.2,1,0.0,0.99,0,none,0,none
4,U001,2024-09-01 00:30:00,126.7,1,0.0,1.12,0,none,0,none


In [3]:
dynamic_df.groupby("patient_id")["glucose"].agg(["mean", "std", "min", "max"]).round(1)


,mean,std,min,max
patient_id,,,,
U001,166.8,21.4,117.3,198.4
U002,126.1,25.5,75.0,160.3
U003,114.2,10.1,91.2,137.0
U004,84.5,7.9,70.0,129.8
U005,99.8,14.5,80.1,133.7
U006,133.4,13.9,102.8,151.1
U007,80.6,7.9,70.0,103.5
U008,134.7,18.7,106.3,176.4
U009,104.0,6.3,91.2,117.8


## 2. Data audit: does this sample actually contain a predictable "spike"?

Before designing a label or a model, we checked whether glucose in this
sample ever *moves* on a clinically relevant timescale. For every reading, we
computed the largest glucose change occurring anywhere in the next 60
minutes -- across the whole file, over every patient, every hour of every
day.


In [4]:
import bisect

def max_rise_within(df, horizon_minutes):
    horizon = pd.Timedelta(minutes=horizon_minutes)
    rises = []
    for _, g in df.groupby("patient_id"):
        g = g.sort_values("timestamp").reset_index(drop=True)
        times = list(g["timestamp"].values)
        glu = g["glucose"].values
        for i in range(len(g)):
            lo = bisect.bisect_right(times, times[i])
            hi = bisect.bisect_right(times, times[i] + horizon)
            if lo < hi:
                rises.append(glu[lo:hi].max() - glu[i])
    return np.array(rises)

rises_60 = max_rise_within(dynamic_df, 60)
print(f"Largest 60-minute rise anywhere in the dataset: {rises_60.max():.1f} mg/dL")
print(f"Mean 60-minute rise: {rises_60.mean():.2f} mg/dL, std: {rises_60.std():.2f} mg/dL")


Largest 60-minute rise anywhere in the dataset: 7.8 mg/dL
Mean 60-minute rise: 1.06 mg/dL, std: 1.29 mg/dL


**Finding:** the largest 60-minute glucose change in the *entire* dataset is
under 12 mg/dL -- smaller than typical CGM sensor noise, and far below what a
90g carbohydrate meal produces in a real patient (often 40-80 mg/dL). A
persistence baseline ("predict no change") would score >99% agreement with
any "future spike" label built directly on this signal, and a Random Forest
regressor trained to forecast raw glucose 60 minutes ahead actually loses to
plain persistence (we verified this: MAE 2.47 vs 1.48 mg/dL).

That is a real, useful finding about the sample -- but it also means the raw
file alone cannot support the project brief's goal of *predicting an adverse
event before it happens*. There is nothing to predict; the signal is already
almost fully determined by its own current value.

The project brief explicitly allows generating synthetic time-series data
when a suitable dynamic dataset is not available. We use that allowance
narrowly and transparently: rather than discarding real data, **Section 4**
adds a documented physiological response layer on top of the real trace.


## 3. Static EHR profile, synthesized from a held-out baseline window

No real EHR extract ships with this sample, so we synthesize one -- but not
arbitrarily. Each patient's **first 3 days** of monitoring are set aside and
used *only* to derive their static profile; the remaining ~6 days are what
the supervised model actually trains and is evaluated on. This mirrors how a
real deployment would work (HbA1c is known *before* live monitoring starts)
and stops the static features from leaking information about the exact rows
being predicted.

Key relationship used: the ADAG study formula linking estimated average
glucose (eAG) to HbA1c, **eAG = 28.7 x A1C - 46.7**, inverted to estimate a
plausible HbA1c from each patient's baseline-window mean glucose. Diabetes
status, medication (stepped therapy: none -> metformin -> metformin+SU ->
insulin+metformin) and previous hyperglycemic episode counts are all derived
from that same HbA1c/glucose baseline via clinically-motivated rules, plus a
small amount of patient-seeded randomness for age/sex/BMI/smoking so the
cohort doesn't look mechanically generated.


In [5]:
static_df = p.build_static_profiles(dynamic_df)
monitoring_df = p.monitoring_window(dynamic_df)
static_df


,patient_id,age,sex,bmi,hba1c,diabetes_status,diabetes_duration_years,medication,previous_hyperglycemic_episodes,smoking_status,response_multiplier,baseline_mean_glucose,baseline_glucose_std
0,U001,45,M,26.4,6.7,True,4,Metformin,17,Current,0.806,141.3,16.3
1,U002,54,F,23.6,4.9,False,0,None,0,Never,0.500,93.8,9.0
2,U003,55,M,27.0,5.1,True,0,Metformin,2,Former,0.518,105.5,5.0
3,U004,48,M,19.0,4.8,False,0,None,2,Never,0.500,83.1,8.9
4,U005,52,F,24.2,4.9,False,0,None,0,Never,0.500,88.4,4.5
5,U006,44,M,25.3,5.6,True,0,Metformin,3,Never,0.608,116.2,6.1
6,U007,47,M,21.1,4.8,False,0,None,1,Former,0.500,77.7,5.5
7,U008,57,M,26.9,5.7,True,2,Metformin,0,Never,0.626,118.5,6.4
8,U009,72,M,25.1,4.8,True,0,Metformin,0,Former,0.500,104.0,7.0
9,U010,38,F,23.9,4.8,False,0,None,0,Former,0.500,82.8,8.6


## 4. Data fusion + physiological augmentation

First, fuse: attach each patient's static profile to every dynamic reading in
their (post-baseline) monitoring window.

Then, the augmentation layer described in Section 2. For every meal (`carbs >
0`) and every exercise bout (`exercise_steps > 0`), we add a causal,
gamma-shaped impulse-response kernel: 0 at the moment of the event, peaking a
fixed number of minutes later, and fully decaying within a bounded window.
Multiple events superpose. The *size* of a meal's response is scaled by the
patient's own synthetic insulin-sensitivity multiplier -- derived from their
HbA1c, so a poorly-controlled patient (high HbA1c) shows a bigger, more
sluggish excursion for the exact same meal than a well-controlled one. This
is the fusion step that actually matters: identical behaviour, different
physiology, different outcome -- which is the entire point of a personalized
Digital Twin.

The original sensor reading is preserved as `glucose_measured`; the modelling
column `glucose` is the augmented signal. Both are kept in every saved
dataset so the difference is always inspectable, never hidden.


In [6]:
fused = p.fuse(monitoring_df, static_df)
augmented = p.simulate_physiological_response(fused)
augmented[["patient_id", "timestamp", "glucose_measured", "glucose", "carbs", "insulin_bolus", "exercise_steps"]].head(8)


,patient_id,timestamp,glucose_measured,glucose,carbs,insulin_bolus,exercise_steps
0,U001,2024-09-04 00:10:00,166.2,166.538566,0,0.0,0
1,U001,2024-09-04 00:20:00,165.2,164.600733,0,0.0,0
2,U001,2024-09-04 00:35:00,166.8,166.079272,0,0.0,0
3,U001,2024-09-04 00:40:00,167.0,165.981871,0,0.0,0
4,U001,2024-09-04 00:45:00,167.7,169.177507,0,0.0,0
5,U001,2024-09-04 00:50:00,168.2,169.993210,0,0.0,0
6,U001,2024-09-04 00:55:00,168.6,168.593541,30,3.0,0
7,U001,2024-09-04 01:10:00,168.3,170.029975,0,0.0,0


In [7]:
# Visualize the augmentation for one patient with several meals
sample_patient = "U004"
g = augmented[augmented["patient_id"] == sample_patient].sort_values("timestamp")
day = g[(g["timestamp"] >= "2024-09-05") & (g["timestamp"] < "2024-09-06")]

fig = go.Figure()
fig.add_trace(go.Scatter(x=day["timestamp"], y=day["glucose_measured"], name="Raw sensor reading",
                          line=dict(color="#90a4ae", dash="dot")))
fig.add_trace(go.Scatter(x=day["timestamp"], y=day["glucose"], name="Augmented Digital Twin signal",
                          line=dict(color="#1565c0", width=2)))
meals = day[day["carbs"] > 0]
fig.add_trace(go.Scatter(x=meals["timestamp"], y=meals["glucose"], mode="markers",
                          marker=dict(color="#c62828", size=9, symbol="triangle-up"), name="Meal event"))
fig.update_layout(title=f"{sample_patient} -- one day, raw vs. physiologically-augmented signal",
                   yaxis_title="Glucose (mg/dL)", height=420)
fig.show()


## 5. Feature engineering (causal / backward-looking only)

Every feature below is computed from data at or before the current timestamp
-- rolling means/std/min/max of glucose (30 min and 2 h windows), rolling
sums of carbs/insulin/exercise over the last hour, minutes since the last
meal, a glucose trend (slope) term, and a cyclical encoding of time of day.
Because nothing here looks forward, the exact same function
(`engineer_features`) is safe to reuse unchanged at live inference time in
the dashboard.


In [8]:
featured = p.engineer_features(augmented)
feature_preview = ["patient_id", "timestamp", "glucose", "glucose_30min_mean", "glucose_slope_30min",
                    "carbs_60min_sum", "insulin_bolus_60min_sum", "exercise_steps_60min_sum", "minutes_since_meal"]
featured[feature_preview].head(8)


,patient_id,timestamp,glucose,glucose_30min_mean,glucose_slope_30min,carbs_60min_sum,insulin_bolus_60min_sum,exercise_steps_60min_sum,minutes_since_meal
0,U001,2024-09-04 00:10:00,166.538566,166.538566,0.000000,0.0,0.0,0.0,999.0
1,U001,2024-09-04 00:20:00,164.600733,165.569650,-0.968917,0.0,0.0,0.0,999.0
2,U001,2024-09-04 00:35:00,166.079272,165.739524,0.339748,0.0,0.0,0.0,999.0
3,U001,2024-09-04 00:40:00,165.981871,165.553959,0.427913,0.0,0.0,0.0,999.0
4,U001,2024-09-04 00:45:00,169.177507,166.459846,2.717661,0.0,0.0,0.0,999.0
5,U001,2024-09-04 00:50:00,169.993210,167.807965,2.185245,0.0,0.0,0.0,999.0
6,U001,2024-09-04 00:55:00,168.593541,167.965080,0.628461,30.0,3.0,0.0,0.0
7,U001,2024-09-04 01:10:00,170.029975,169.448558,0.581417,30.0,3.0,0.0,15.0


## 6. Forecast target and honest baselines

The prediction target is **glucose 60 minutes from now** (regression, not a
binary spike label -- Section 2 explained why a binary label on this kind of
low-volatility trace degenerates into near-persistence classification).

Two zero-training baselines are computed alongside it, because a model that
can't beat "predict no change" has no business being deployed:

- **Persistence**: forecast = current glucose.
- **Trend extrapolation**: forecast = current glucose + current 30-minute slope.


In [9]:
targeted = p.build_forecast_target(featured)
splits = p.time_based_split(targeted)
for name, d in splits.items():
    print(f"{name:5s}: n={len(d):5d}  date range {d['timestamp'].min()} -> {d['timestamp'].max()}")


train: n= 7131  date range 2024-09-04 00:00:00 -> 2024-09-08 02:40:00
val  : n= 1491  date range 2024-09-08 03:35:00 -> 2024-09-09 00:00:00
test : n= 1544  date range 2024-09-09 00:55:00 -> 2024-09-09 21:30:00


Splits are **purged**: each patient's timeline is cut chronologically into
train (70%) / val (15%) / test (15%), and any row whose 60-minute-ahead
target would reach past its split's boundary is dropped rather than kept --
otherwise a training row could quietly peek at data technically inside the
validation window.


## 7. Model training and comparison

Two candidate models, both trained on the fused static + dynamic feature set:

- **Ridge regression** -- linear, interpretable, a sensible first model.
- **Random Forest regressor** -- nonlinear, can capture interaction effects
  (e.g. "a big meal matters more for a high-HbA1c patient").

Both are compared against the persistence and trend-extrapolation baselines
on train/val/test, **and** on the subset of test rows with a meal or exercise
event in the last hour -- the rows where fused behavioural + static features
should matter most, if they matter at all.


In [10]:
models = p.build_models()
fitted, forecasts = {}, {}
rows = []

baseline_table = {split: p.baseline_metrics(d) for split, d in splits.items()}
for split_name, metrics in baseline_table.items():
    for baseline_name, m in metrics.items():
        rows.append({"model": baseline_name, "split": split_name, "mae": m["mae"], "rmse": m["rmse"], "r2": m["r2"]})

for name, pipe in models.items():
    pipe.fit(splits["train"][p.ALL_FEATURES], splits["train"][p.TARGET_COL])
    fitted[name] = pipe
    forecasts[name] = {}
    for split_name in ("train", "val", "test"):
        d = splits[split_name]
        pred = pipe.predict(d[p.ALL_FEATURES])
        forecasts[name][split_name] = pred
        m = p.evaluate_regression(d[p.TARGET_COL], pred)
        rows.append({"model": name, "split": split_name, "mae": m["mae"], "rmse": m["rmse"], "r2": m["r2"]})

comparison = pd.DataFrame(rows).pivot(index="model", columns="split", values="mae").round(2)
comparison = comparison[["train", "val", "test"]]
comparison.columns = [f"MAE ({c})" for c in comparison.columns]
comparison.sort_values("MAE (val)")


,MAE (train),MAE (val),MAE (test)
model,,,
ridge,11.05,13.28,14.58
persistence,12.75,14.04,13.91
trend_extrapolation,13.03,14.29,14.13
random_forest,6.98,14.68,15.91


In [11]:
best_name = min(("ridge", "random_forest"), key=lambda n: p.evaluate_regression(
    splits["val"][p.TARGET_COL], forecasts[n]["val"])["mae"])
best_pipeline = fitted[best_name]
print("Selected model (lowest validation MAE):", best_name)

meal_metrics = p.meal_or_exercise_subset_metrics(splits["test"], forecasts[best_name]["test"])
print("\nOn test rows following a meal or exercise bout in the last hour (the rows that matter clinically):")
print(f"  n = {meal_metrics['n']}")
print(f"  {best_name} MAE:      {meal_metrics['model']['mae']:.2f} mg/dL")
print(f"  persistence MAE:      {meal_metrics['persistence']['mae']:.2f} mg/dL")


Selected model (lowest validation MAE): ridge

On test rows following a meal or exercise bout in the last hour (the rows that matter clinically):
  n = 730
  ridge MAE:      14.52 mg/dL
  persistence MAE:      15.67 mg/dL


**Reading this honestly:** on the *full* test set, persistence remains
competitive -- most of the time nothing dynamic is happening, and raw
continuity is a strong predictor by itself. But specifically on the rows
that follow a meal or exercise event -- the clinically important moments
where a doctor would actually want the twin's help -- the fused model beats
persistence. That is where personalization (static risk profile + recent
behaviour) earns its keep, and reporting it separately is more honest than a
single blended metric that could hide a model that only wins on the boring
majority of rows.


## 8. From point forecast to hyperglycemia-risk probability

A doctor dashboard needs a probability, not just a number in mg/dL. Assuming
approximately normal forecast residuals (estimated on the *validation* set,
never on test), the probability that the true future glucose exceeds the
140 mg/dL hyperglycemia threshold is the upper tail of a normal distribution
centered on the forecast:

$$P(\text{glucose}_{t+60} \ge 140) = 1 - \Phi\left(\frac{140 - \hat{y}}{\sigma_{\text{resid}}}\right)$$


In [12]:
resid_std = p.residual_std(splits["val"][p.TARGET_COL], forecasts[best_name]["val"])
print(f"Validation-set residual std ({best_name}): {resid_std:.2f} mg/dL")

test_scored = splits["test"].copy()
test_scored["predicted_glucose_60min"] = forecasts[best_name]["test"]
test_scored["predicted_hyperglycemia_risk"] = p.forecast_to_risk_probability(
    test_scored["predicted_glucose_60min"], resid_std)

test_scored.groupby("patient_id")["predicted_hyperglycemia_risk"].mean().sort_values(ascending=False).round(3)


Validation-set residual std (ridge): 16.83 mg/dL


patient_id
U001    0.989
U008    0.663
U006    0.231
U003    0.211
U002    0.061
U005    0.055
U009    0.015
U007    0.000
U004    0.000
U010    0.000
Name: predicted_hyperglycemia_risk, dtype: float64

The risk score varies dramatically and sensibly by patient: poorly-controlled
patients sit near 90-100% risk almost continuously, well-controlled patients
sit near 0%, and a couple of intermediate patients land in a genuinely
informative 20-40% band -- exactly the patients a clinician would want
flagged for a closer look, and exactly the personalization a purely
population-level model could never produce.


## 9. Explainability: two separate questions, not one blended ranking

Static features (HbA1c, prior episodes, age, BMI, diabetes duration) never
change for a given patient, so mixing them into a single "why" ranking with
real-time features lets a permanent risk factor drown out what actually
changed in the last hour. We split explainability into two panels that
answer two different clinical questions:

- **`explain_static_risk_factors`** -- how does this patient's baseline
  profile compare to a population-typical patient? (occlusion against the
  *population* median)
- **`explain_dynamic_drivers`** -- what's different about *this moment*
  compared to this *same patient's* own typical moment? (occlusion against
  the *patient's own* median, not the population's)

Both are computed by single-feature occlusion (replace one feature with a
reference value, measure the change in forecast) -- simple, dependency-free,
and reused as-is by the dashboard. Note: HbA1c and prior-episode count are
correlated by construction (both derive from the same baseline glucose
history), so occlusion magnitudes should be read as a *ranking*, not
precisely additive mg/dL contributions.


In [13]:
example_patient = "U001"
p_df = targeted[targeted["patient_id"] == example_patient].sort_values("timestamp").reset_index(drop=True)
row = p_df.iloc[-1]

print("Baseline risk factors (vs. population):")
display(p.explain_static_risk_factors(best_pipeline, row, population_reference=targeted, top_k=5))

print("\nRight-now drivers (vs. this patient's own typical moment):")
display(p.explain_dynamic_drivers(best_pipeline, row, patient_history=p_df, top_k=5))


Baseline risk factors (vs. population):


,feature,forecast_contribution_mgdl
4,previous_hyperglycemic_episodes,104.878596
2,hba1c,87.705973
0,age,-8.310956
3,diabetes_duration_years,2.886748
1,bmi,-1.876547



Right-now drivers (vs. this patient's own typical moment):


,feature,forecast_contribution_mgdl
7,glucose_slope_30min,2.973318
2,glucose_30min_std,1.903802
5,glucose_2h_mean,-1.833299
1,glucose_30min_mean,-1.204165
6,glucose_2h_std,1.166228


## 10. Regenerate the canonical artifacts

Everything above walked through the pipeline step-by-step for narrative and
inspection. `pipeline.run()` executes the exact same logic end-to-end and
writes the artifacts the dashboard actually loads:

- `data/processed/patient_twin_dataset.csv` -- fused, augmented, featured,
  labeled dataset with the deployed model's forecasts and risk scores.
- `data/processed/static_patient_profiles.csv`
- `models/deployed_model.joblib`, `models/ridge.joblib`, `models/random_forest.joblib`
- `models/feature_columns.json`, `models/metrics.json`

Run this cell whenever the pipeline code changes, then launch the dashboard
with `streamlit run dashboard/app.py` from the project root.


In [14]:
metrics = p.run()
print("Selected model:", metrics["selected_model"])
print("Validation residual std:", round(metrics["residual_std_val"], 2), "mg/dL")


Selected model: ridge
Validation residual std: 16.83 mg/dL


## 11. Limitations (read before presenting this as more than a proof-of-concept)

- **The dynamic excursions are partly synthetic.** The real CGM trace in this
  sample is very low-volatility; Section 2/4 documents exactly how and why a
  physiologically-motivated response layer was added on top of it. A
  production system would need real high-resolution CGM data (e.g. the full
  GlucoBench release, or a clinically consented feed) to validate that these
  effect sizes and the model's edge over persistence hold up.
- **10 patients, ~6 usable days each.** Small enough that per-patient
  idiosyncrasies (one very poorly-controlled patient, several very
  well-controlled ones) can dominate aggregate metrics. Treat every number
  here as a proof-of-concept signal, not a validated clinical result.
- **The static EHR profile is synthesized, not real.** It is derived from the
  patient's own baseline-window glucose statistics using clinically-grounded
  formulas (ADAG, stepped therapy), but it is not a substitute for an actual
  EHR extract.
- **Risk probabilities assume normal, homoscedastic residuals.** A real
  deployment should validate calibration (e.g. a reliability diagram) rather
  than trust the Gaussian assumption blindly.
- **This is decision support, not a diagnosis.** Every risk score and
  "what-if" scenario in the dashboard is a modeling estimate, not a
  clinical instruction.
